In [ ]:
"""I3TruthExtractorPONE: truth-level extractor for PONE simulations."""

import numpy as np
import matplotlib.path as mpath
from scipy.spatial import ConvexHull, Delaunay
from typing import TYPE_CHECKING, Any, Dict, List, Optional, Tuple

from .i3extractor import I3Extractor
from .utilities.frames import (
    frame_is_montecarlo,
    frame_is_noise,
)
from graphnet.utilities.imports import has_icecube_package

if has_icecube_package() or TYPE_CHECKING:
    from icecube import (  # noqa: F401
        dataclasses,
        icetray,
        phys_services,
        dataio,
        LeptonInjector,
    )  # pyright: reportMissingImports=false


class I3TruthExtractorPONE(I3Extractor):
    """Truth-level extractor for PONE simulations.

    Compared to I3TruthExtractor, this class:
      - Derives detector boundaries dynamically from the GCD file instead
        of relying on hardcoded IceCube coordinates.
      - Does not filter on sub_event_stream (no InIceSplit/Final requirement),
        since PONE data does not use that frame splitting convention.
    """

    def __init__(
        self,
        name: str = "truth",
        mctree: Optional[str] = "I3MCTree",
        extend_boundary: Optional[float] = 0.0,
        exclude: list = [None],
    ):
        """Construct I3TruthExtractorPONE.

        Args:
            name: Name of the I3Extractor instance.
            mctree: Name of the MC tree to use for truth values.
            extend_boundary: Distance in metres to extend the convex hull of
                the detector when defining starting events. Defaults to 0.
            exclude: List of keys to exclude from the extracted data.
        """
        super().__init__(name, exclude=exclude)

        # Borders cannot be computed here because the GCD file has not been
        # loaded yet. They will be set in set_gcd() once sensor positions
        # are available.
        self._borders = None
        self._extend_boundary = extend_boundary
        self._mctree = mctree

    def set_gcd(self, i3_file: str, gcd_file: Optional[str] = None) -> None:
        """Load GCD file and derive detector geometry.

        Calls the parent implementation to populate self._gcd_dict, then
        computes detector boundaries and the Delaunay triangulation used for
        vertex containment checks — all from actual sensor positions rather
        than hardcoded coordinates.

        Args:
            i3_file: Path to the i3 file being converted.
            gcd_file: Path to the GCD file. If None, the method will look for
                G and C frames inside i3_file.
        """
        super().set_gcd(i3_file=i3_file, gcd_file=gcd_file)

        # Collect 3D sensor positions from the GCD
        coordinates = np.array([
            [g.position.x, g.position.y, g.position.z]
            for g in self._gcd_dict.values()
        ])

        # Optionally expand the boundary outward from the centroid.
        # This is useful when you want to include events that interact just
        # outside the instrumented volume.
        if self._extend_boundary != 0.0:
            center = np.mean(coordinates, axis=0)
            d = coordinates - center
            norms = np.linalg.norm(d, axis=1, keepdims=True)
            dn = d / norms
            coordinates = coordinates + dn * self._extend_boundary

        # Build the 3D convex hull and Delaunay triangulation used by
        # _contained_vertex() to test whether an interaction vertex lies
        # inside the detector.
        hull = ConvexHull(coordinates)
        self.hull = hull
        self.delaunay = Delaunay(coordinates[hull.vertices])

        # Derive the 2D (XY) boundary polygon and Z range used by
        # _muon_stopped() to test whether a muon's final position lies
        # within the fiducial volume.
        xy = coordinates[:, :2]
        hull_2d = ConvexHull(xy)
        border_xy = xy[hull_2d.vertices]
        border_z = np.array([coordinates[:, 2].min(), coordinates[:, 2].max()])
        self._borders = [border_xy, border_z]

    def __call__(
        self, frame: "icetray.I3Frame", padding_value: Any = -1
    ) -> Dict[str, Any]:
        """Extract truth-level information from a physics frame."""
        is_mc = frame_is_montecarlo(frame, self._mctree)
        is_noise = frame_is_noise(frame, self._mctree)
        sim_type = self._find_data_type(is_mc, self._i3_file, frame)

        output = {
            "energy": padding_value,
            "position_x": padding_value,
            "position_y": padding_value,
            "position_z": padding_value,
            "azimuth": padding_value,
            "zenith": padding_value,
            "pid": padding_value,
            "event_time": frame["I3EventHeader"].start_time.utc_daq_time,
            "sim_type": sim_type,
            "interaction_type": padding_value,
            "elasticity": padding_value,
            "RunID": frame["I3EventHeader"].run_id,
            "SubrunID": frame["I3EventHeader"].sub_run_id,
            "EventID": frame["I3EventHeader"].event_id,
            "SubEventID": frame["I3EventHeader"].sub_event_id,
            "dbang_decay_length": padding_value,
            "track_length": padding_value,
            "stopped_muon": padding_value,
            "energy_track": padding_value,
            "energy_cascade": padding_value,
            "inelasticity": padding_value,
            "DeepCoreFilter_13": padding_value,
            "CascadeFilter_13": padding_value,
            "MuonFilter_13": padding_value,
            "OnlineL2Filter_17": padding_value,
            "L3_oscNext_bool": padding_value,
            "L4_oscNext_bool": padding_value,
            "L5_oscNext_bool": padding_value,
            "L6_oscNext_bool": padding_value,
            "L7_oscNext_bool": padding_value,
            "is_starting": padding_value,
        }

        # Note: the InIceSplit / Final sub_event_stream filter present in
        # I3TruthExtractor is intentionally omitted here. PONE data does not
        # use that frame splitting convention.

        if "FilterMask" in frame:
            if "DeepCoreFilter_13" in frame["FilterMask"]:
                output["DeepCoreFilter_13"] = int(
                    bool(frame["FilterMask"]["DeepCoreFilter_13"])
                )
            if "CascadeFilter_13" in frame["FilterMask"]:
                output["CascadeFilter_13"] = int(
                    bool(frame["FilterMask"]["CascadeFilter_13"])
                )
            if "MuonFilter_13" in frame["FilterMask"]:
                output["MuonFilter_13"] = int(
                    bool(frame["FilterMask"]["MuonFilter_13"])
                )
            if "OnlineL2Filter_17" in frame["FilterMask"]:
                output["OnlineL2Filter_17"] = int(
                    bool(frame["FilterMask"]["OnlineL2Filter_17"])
                )

        elif "DeepCoreFilter_13" in frame:
            output["DeepCoreFilter_13"] = int(bool(frame["DeepCoreFilter_13"]))

        if "L3_oscNext_bool" in frame:
            output["L3_oscNext_bool"] = int(bool(frame["L3_oscNext_bool"]))

        if "L4_oscNext_bool" in frame:
            output["L4_oscNext_bool"] = int(bool(frame["L4_oscNext_bool"]))

        if "L5_oscNext_bool" in frame:
            output["L5_oscNext_bool"] = int(bool(frame["L5_oscNext_bool"]))

        if "L6_oscNext_bool" in frame:
            output["L6_oscNext_bool"] = int(bool(frame["L6_oscNext_bool"]))

        if "L7_oscNext_bool" in frame:
            output["L7_oscNext_bool"] = int(bool(frame["L7_oscNext_bool"]))

        if is_mc and (not is_noise):
            (
                MCInIcePrimary,
                interaction_type,
                elasticity,
            ) = self._get_primary_particle_interaction_type_and_elasticity(
                frame, sim_type
            )

            try:
                (
                    energy_track,
                    energy_cascade,
                    inelasticity,
                ) = self._get_primary_track_energy_and_inelasticity(frame)
            except RuntimeError:
                # Track energy calculation fails for some northern tracks where
                # the Hadrons particle has no mass implemented.
                energy_track, energy_cascade, inelasticity = (
                    padding_value,
                    padding_value,
                    padding_value,
                )

            output.update(
                {
                    "energy": MCInIcePrimary.energy,
                    "position_x": MCInIcePrimary.pos.x,
                    "position_y": MCInIcePrimary.pos.y,
                    "position_z": MCInIcePrimary.pos.z,
                    "azimuth": MCInIcePrimary.dir.azimuth,
                    "zenith": MCInIcePrimary.dir.zenith,
                    "pid": MCInIcePrimary.pdg_encoding,
                    "interaction_type": interaction_type,
                    "elasticity": elasticity,
                    "dbang_decay_length": self._extract_dbang_decay_length(
                        frame, padding_value
                    ),
                    "energy_track": energy_track,
                    "energy_cascade": energy_cascade,
                    "inelasticity": inelasticity,
                }
            )

            if abs(output["pid"]) == 13:
                output.update({"track_length": MCInIcePrimary.length})
                muon_final = self._muon_stopped(output, self._borders)
                output.update(
                    {
                        # For muons, position_xyz is updated to the final
                        # stopping position rather than the interaction vertex.
                        "position_x": muon_final["x"],
                        "position_y": muon_final["y"],
                        "position_z": muon_final["z"],
                        "stopped_muon": muon_final["stopped"],
                    }
                )

            output.update({"is_starting": self._contained_vertex(output)})

        return output

    def _extract_dbang_decay_length(
        self, frame: "icetray.I3Frame", padding_value: float = -1
    ) -> float:
        """Extract the decay length for double-bang (HNL) events.

        Args:
            frame: Physics frame containing MC record.
            padding_value: Value returned when the decay length cannot be
                determined.

        Returns:
            Decay length in metres, or padding_value on failure.
        """
        mctree = frame[self._mctree]
        try:
            p_true = mctree.primaries[0]
            p_daughters = mctree.get_daughters(p_true)
            if len(p_daughters) == 2:
                for p_daughter in p_daughters:
                    if p_daughter.type == dataclasses.I3Particle.Hadrons:
                        casc_0_true = p_daughter
                    else:
                        hnl_true = p_daughter
                hnl_daughters = mctree.get_daughters(hnl_true)
            else:
                decay_length = padding_value
                hnl_daughters = []

            if len(hnl_daughters) > 0:
                for count_hnl_daughters, hnl_daughter in enumerate(
                    hnl_daughters
                ):
                    if not count_hnl_daughters:
                        casc_1_true = hnl_daughter
                    else:
                        assert casc_1_true.pos == hnl_daughter.pos
                        casc_1_true.energy = (
                            casc_1_true.energy + hnl_daughter.energy
                        )
                decay_length = (
                    phys_services.I3Calculator.distance(
                        casc_0_true, casc_1_true
                    )
                    / icetray.I3Units.m
                )
            else:
                decay_length = padding_value

            return decay_length
        except:  # noqa: E722
            return padding_value

    def _muon_stopped(
        self,
        truth: Dict[str, Any],
        borders: List[np.ndarray],
        shrink_horizontally: float = 100.0,
        shrink_vertically: float = 100.0,
    ) -> Dict[str, Any]:
        """Compute the muon's final position and whether it stopped inside
        the fiducial volume.

        The final position is calculated by propagating the muon from its
        starting position along its direction vector by track_length metres.

        IMPORTANT: The result is stored as position_x/y/z in the output dict,
        overwriting the interaction vertex — consistent with how I3TruthExtractor
        handles muons.

        Args:
            truth: Dictionary of already extracted truth-level information.
            borders: [border_xy, border_z] where border_xy is an (N, 2) array
                of polygon vertices in the XY plane and border_z is [z_min, z_max].
            shrink_horizontally: Inward shrink of the XY polygon in metres.
                Defaults to 100.
            shrink_vertically: Inward shrink in the Z direction in metres.
                Defaults to 100.

        Returns:
            Dictionary with keys x, y, z (final position) and stopped (bool).
        """
        border = mpath.Path(borders[0])

        start_pos = np.array(
            [truth["position_x"], truth["position_y"], truth["position_z"]]
        )

        # Propagate along the direction vector. The -1 factor is needed because
        # zenith/azimuth in IceCube convention point toward the particle origin,
        # not its direction of travel.
        travel_vec = -1 * np.array(
            [
                truth["track_length"]
                * np.cos(truth["azimuth"])
                * np.sin(truth["zenith"]),
                truth["track_length"]
                * np.sin(truth["azimuth"])
                * np.sin(truth["zenith"]),
                truth["track_length"] * np.cos(truth["zenith"]),
            ]
        )

        end_pos = start_pos + travel_vec

        stopped_xy = border.contains_point(
            (end_pos[0], end_pos[1]), radius=-shrink_horizontally
        )
        stopped_z = (end_pos[2] > borders[1][0] + shrink_vertically) * (
            end_pos[2] < borders[1][1] - shrink_vertically
        )

        return {
            "x": end_pos[0],
            "y": end_pos[1],
            "z": end_pos[2],
            "stopped": (stopped_xy * stopped_z),
        }

    def _get_primary_particle_interaction_type_and_elasticity(
        self,
        frame: "icetray.I3Frame",
        sim_type: str,
        padding_value: float = -1.0,
    ) -> Tuple[Any, int, float]:
        """Return the primary MC particle, interaction type, and elasticity.

        Args:
            frame: Physics frame containing MC record.
            sim_type: Simulation type string (e.g. 'LeptonInjector', 'NuGen').
            padding_value: Fallback value used when a quantity cannot be read.

        Returns:
            Tuple of (MCInIcePrimary, interaction_type, elasticity) where
            interaction_type is 1 (CC), 2 (NC), or padding_value.
        """
        if sim_type != "noise":
            try:
                MCInIcePrimary = frame["MCInIcePrimary"]
            except KeyError:
                MCInIcePrimary = frame[self._mctree][0]
            if MCInIcePrimary.energy != MCInIcePrimary.energy:
                # NaN check — occurs for some muons where the first MCTree
                # entry has corrupted energy. The second entry is always valid.
                MCInIcePrimary = frame[self._mctree][1]
        else:
            MCInIcePrimary = None

        if sim_type == "LeptonInjector":
            event_properties = frame["EventProperties"]
            final_state_1 = event_properties.finalType1
            if final_state_1 in [
                dataclasses.I3Particle.NuE,
                dataclasses.I3Particle.NuMu,
                dataclasses.I3Particle.NuTau,
                dataclasses.I3Particle.NuEBar,
                dataclasses.I3Particle.NuMuBar,
                dataclasses.I3Particle.NuTauBar,
            ]:
                interaction_type = 2  # NC
            else:
                interaction_type = 1  # CC

            elasticity = 1 - event_properties.finalStateY

        else:
            try:
                interaction_type = frame["I3MCWeightDict"]["InteractionType"]
            except KeyError:
                interaction_type = int(padding_value)

            try:
                elasticity = 1 - frame["I3MCWeightDict"]["BjorkenY"]
            except KeyError:
                elasticity = padding_value

        return MCInIcePrimary, interaction_type, elasticity

    def _get_primary_track_energy_and_inelasticity(
        self,
        frame: "icetray.I3Frame",
    ) -> Tuple[float, float, float]:
        """Compute track energy, cascade energy, and inelasticity from primary.

        Args:
            frame: Physics frame containing MC record.

        Returns:
            Tuple of (energy_track, energy_cascade, inelasticity).
        """
        mc_tree = frame[self._mctree]
        primary = mc_tree.primaries[0]
        daughters = mc_tree.get_daughters(primary)

        tracks = [
            d for d in daughters
            if str(d.shape) in ("StartingTrack", "Dark")
        ]

        energy_total = primary.total_energy
        energy_track = sum(t.total_energy for t in tracks)
        energy_cascade = energy_total - energy_track
        inelasticity = 1.0 - energy_track / energy_total

        return energy_track, energy_cascade, inelasticity

    def _find_data_type(
        self, mc: bool, input_file: str, frame: "icetray.I3Frame"
    ) -> str:
        """Determine the simulation or data type from the file path and frame.

        Args:
            mc: Whether the input file is Monte Carlo simulation.
            input_file: Path to the i3 file.
            frame: Physics frame, used as fallback when path is uninformative.

        Returns:
            A string identifying the data type, e.g. 'LeptonInjector', 'NuGen',
            'muongun', 'corsika', 'genie', 'noise', or 'data'.
        """
        if not mc:
            sim_type = "data"
        elif "muon" in input_file:
            sim_type = "muongun"
        elif "corsika" in input_file:
            sim_type = "corsika"
        elif "genie" in input_file or "nu" in input_file.lower():
            sim_type = "genie"
        elif "noise" in input_file:
            sim_type = "noise"
        elif frame.Has("EventProperties") or frame.Has(
            "LeptonInjectorProperties"
        ):
            sim_type = "LeptonInjector"
        elif frame.Has("I3MCWeightDict"):
            sim_type = "NuGen"
        else:
            raise NotImplementedError("Could not determine data type.")
        return sim_type

    def _contained_vertex(self, truth: Dict[str, Any]) -> bool:
        """Check whether an interaction vertex lies inside the detector volume.

        Uses the Delaunay triangulation built from GCD sensor positions in
        set_gcd().

        Args:
            truth: Dictionary of already extracted truth-level information.

        Returns:
            True if the vertex is inside the detector, False otherwise.
        """
        vertex = np.array(
            [truth["position_x"], truth["position_y"], truth["position_z"]]
        )
        return self.delaunay.find_simplex(vertex) >= 0

graphnet [MainProcess] WARNING  2025-12-22 12:44:47 - <module> - `km3net` not available. Some functionality may be missing.


/home/kbas/.local/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


graphnet [MainProcess] WARNING  2025-12-22 12:44:52 - <module> - `jammy_flows` not available. Normalizing Flow functionality is missing.


In [2]:
from graphnet.data.extractors.icecube.utilities.frames import frame_is_montecarlo, frame_is_noise
from icecube import dataio

In [3]:
DATA_PATH = "/project/def-nahee/kbas/POM_Response/pom_response_batch_000.i3.gz"
GCD_PATH = "/project/6008051/pone_simulation/GCD_Library/PONE_800mGrid.i3.gz"

In [4]:
my_truth_extractor = I3TruthExtractorPONE(mctree = 'I3MCTree_postprop')

graphnet [MainProcess] INFO     2025-12-22 12:44:52 - I3TruthExtractorPONE.__init__ - Writing log to logs/graphnet_20251222-124452.log


In [5]:
my_truth_extractor._mctree

'I3MCTree_postprop'

In [6]:
my_truth_extractor._borders

[array([[-256.14001465, -521.08001709],
        [-132.80000305, -501.45001221],
        [  -9.13000011, -481.73999023],
        [ 114.38999939, -461.98999023],
        [ 237.77999878, -442.42001343],
        [ 361.        , -422.82998657],
        [ 405.82998657, -306.38000488],
        [ 443.6000061 , -194.16000366],
        [ 500.42999268,  -58.45000076],
        [ 544.07000732,   55.88999939],
        [ 576.36999512,  170.91999817],
        [ 505.26998901,  257.88000488],
        [ 429.76000977,  351.01998901],
        [ 338.44000244,  463.72000122],
        [ 224.58000183,  432.3500061 ],
        [ 101.04000092,  412.79000854],
        [  22.11000061,  509.5       ],
        [-101.05999756,  490.22000122],
        [-224.08999634,  470.85998535],
        [-347.88000488,  451.51998901],
        [-392.38000488,  334.23999023],
        [-437.04000854,  217.80000305],
        [-481.6000061 ,  101.38999939],
        [-526.63000488,  -15.60000038],
        [-570.90002441, -125.13999939],


In [7]:
my_truth_extractor._borders[0]

array([[-256.14001465, -521.08001709],
       [-132.80000305, -501.45001221],
       [  -9.13000011, -481.73999023],
       [ 114.38999939, -461.98999023],
       [ 237.77999878, -442.42001343],
       [ 361.        , -422.82998657],
       [ 405.82998657, -306.38000488],
       [ 443.6000061 , -194.16000366],
       [ 500.42999268,  -58.45000076],
       [ 544.07000732,   55.88999939],
       [ 576.36999512,  170.91999817],
       [ 505.26998901,  257.88000488],
       [ 429.76000977,  351.01998901],
       [ 338.44000244,  463.72000122],
       [ 224.58000183,  432.3500061 ],
       [ 101.04000092,  412.79000854],
       [  22.11000061,  509.5       ],
       [-101.05999756,  490.22000122],
       [-224.08999634,  470.85998535],
       [-347.88000488,  451.51998901],
       [-392.38000488,  334.23999023],
       [-437.04000854,  217.80000305],
       [-481.6000061 ,  101.38999939],
       [-526.63000488,  -15.60000038],
       [-570.90002441, -125.13999939],
       [-492.42999268, -2

In [8]:
my_truth_extractor._borders[1]

array([-512.82,  524.56])

In [9]:
my_truth_extractor._extend_boundary

0.0

In [10]:
my_truth_extractor._i3_file

''

In [11]:
my_truth_extractor._is_corsika

False

In [12]:
my_truth_extractor._gcd_file

''

In [13]:
my_truth_extractor._gcd_dict

{}

In [14]:
my_truth_extractor._calibration

In [15]:
my_truth_extractor._extractor_name

'truth'

In [16]:
my_truth_extractor._exclude

[None]

In [17]:
my_truth_extractor.hull

AttributeError: 'I3TruthExtractorPONE' object has no attribute 'hull'

In [18]:
my_truth_extractor.delaunay

AttributeError: 'I3TruthExtractorPONE' object has no attribute 'delaunay'

In [19]:
my_truth_extractor.set_gcd(i3_file=DATA_PATH, gcd_file=GCD_PATH)

In [20]:
my_truth_extractor._i3_file

'/project/def-nahee/kbas/POM_Response/pom_response_batch_000.i3.gz'

In [21]:
my_truth_extractor._gcd_file

'/project/6008051/pone_simulation/GCD_Library/PONE_800mGrid.i3.gz'

In [22]:
my_truth_extractor._gcd_dict

In [23]:
my_truth_extractor._calibration

In [24]:
my_truth_extractor.hull

In [25]:
print(my_truth_extractor.hull)

In [26]:
my_truth_extractor.delaunay

In [27]:
print(my_truth_extractor.delaunay)

In [28]:
data_file = dataio.I3File(DATA_PATH)
frame = data_file.pop_physics()

In [29]:
frame_is_montecarlo(frame)

True

In [30]:
my_truth_extractor._find_data_type(frame_is_montecarlo(frame), DATA_PATH, frame )

'LeptonInjector'

In [31]:
my_truth_extractor(frame)

{'energy': 119.30066500027469,
 'position_x': 46.768786680196,
 'position_y': 344.60676053887323,
 'position_z': 276.39457112073273,
 'azimuth': 1.378986580623348,
 'zenith': 2.1294179528330694,
 'pid': 14,
 'event_time': 0,
 'sim_type': 'LeptonInjector',
 'interaction_type': 1,
 'elasticity': 0.815154748398404,
 'RunID': 0,
 'SubrunID': 4294967295,
 'EventID': 1,
 'SubEventID': 0,
 'dbang_decay_length': -1,
 'track_length': -1,
 'stopped_muon': -1,
 'energy_track': 97.35410453129462,
 'energy_cascade': 21.946560468980067,
 'inelasticity': 0.18396008495786287,
 'DeepCoreFilter_13': -1,
 'CascadeFilter_13': -1,
 'MuonFilter_13': -1,
 'OnlineL2Filter_17': -1,
 'L3_oscNext_bool': -1,
 'L4_oscNext_bool': -1,
 'L5_oscNext_bool': -1,
 'L6_oscNext_bool': -1,
 'L7_oscNext_bool': -1,
 'is_starting': True}

In [32]:
my_truth_extractor._get_primary_particle_interaction_type_and_elasticity(frame, my_truth_extractor._find_data_type(frame_is_montecarlo(frame), DATA_PATH, frame ) )

(<icecube._dataclasses.I3Particle at 0x14e754e67a70>, 1, 0.815154748398404)

In [33]:
my_truth_extractor._get_primary_track_energy_and_inelasticity(frame) 
## need to use I3MCTreePostProp here. otherwise, the three stuff here will be wrong!
## change the default initialization for the MCTree in your class above. But before, check it if it safe/correct to change it.
## modify the class I3TruthExtractor_PONE_by_LeptonInjector in the py file. not here!

(97.35410453129462, 21.946560468980067, 0.18396008495786287)

In [34]:
my_truth_extractor._contained_vertex(my_truth_extractor(frame)) 

True

In [35]:
## there are 1-2 stuff that I did not check yet. Check them too!

In [36]:
## edit your PONE Truth Extractor

In [37]:
my_truth_extractor_with_exclude = I3TruthExtractorPONE(mctree = 'I3MCTree_postprop', 
                                                       exclude=['L7_oscNext_bool', 'L6_oscNext_bool',
                                                               'L5_oscNext_bool', 'L4_oscNext_bool',
                                                               'L3_oscNext_bool',
                                                               'OnlineL2Filter_17','MuonFilter_13',
                                                               'CascadeFilter_13','DeepCoreFilter_13'])

In [38]:
my_truth_extractor_with_exclude.set_gcd(i3_file=DATA_PATH, gcd_file=GCD_PATH)

In [39]:
data_file = dataio.I3File(DATA_PATH)
frame = data_file.pop_physics()

In [40]:
my_truth_extractor_with_exclude(frame)

{'energy': 119.30066500027469,
 'position_x': 46.768786680196,
 'position_y': 344.60676053887323,
 'position_z': 276.39457112073273,
 'azimuth': 1.378986580623348,
 'zenith': 2.1294179528330694,
 'pid': 14,
 'event_time': 0,
 'sim_type': 'LeptonInjector',
 'interaction_type': 1,
 'elasticity': 0.815154748398404,
 'RunID': 0,
 'SubrunID': 4294967295,
 'EventID': 1,
 'SubEventID': 0,
 'dbang_decay_length': -1,
 'track_length': -1,
 'stopped_muon': -1,
 'energy_track': 97.35410453129462,
 'energy_cascade': 21.946560468980067,
 'inelasticity': 0.18396008495786287,
 'is_starting': True}